## [실습 과제] Day 2 - Part 5: 전이학습과 파인튜닝

### 목표
이번 과제에서는 튜토리얼에서 배운 전이학습과 미세조정 기법을 `다른 데이터셋`과 `다른 사전 학습 모델`에 적용하여 실전 능력을 강화하는 것을 목표로 합니다.

여러분은 딥러닝 엔지니어로서 'Intel Image Classification' 데이터셋을 분류하는 모델을 구축하는 임무를 맡았습니다. 이 데이터셋은 'buildings', 'forest', 'glacier', 'mountain', 'sea', 'street' 6개의 클래스로 구성되어 있습니다.

`요구사항:`
1. `torchvision.models`에서 `MobileNet_V2` 사전 학습 모델을 사용하세요.
2. `특징 추출(Feature Extraction)` 전략으로 모델을 먼저 학습시키세요.
3. `미세조정(Fine-Tuning)` 전략으로 모델의 성능을 추가로 개선해보세요.
4. 학습 과정의 손실 및 정확도 변화를 `plotly`로 시각화하세요.
5. 최종 모델의 성능을 `혼동 행렬(Confusion Matrix)` 로 시각화하고 분석하세요.

### Part 1: 데이터 준비 (Data Preparation)

먼저, 데이터를 다운로드하고 PyTorch가 사용할 수 있는 형태로 준비합니다. Kaggle API 등을 사용하거나 직접 다운로드하여 아래와 같은 폴더 구조를 만드세요.

```
seg_train/
  ├── buildings/
  ├── forest/
  ...
seg_test/
  ├── buildings/
  ├── forest/
  ...
```

In [1]:
import kaggle
kaggle.api.authenticate()
data_dir = "../../datasets/dl/intel-image-classification"
kaggle.api.dataset_download_files("puneet6060/intel-image-classification", path=data_dir, unzip=True)

Dataset URL: https://www.kaggle.com/datasets/puneet6060/intel-image-classification


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix

# --- [TODO] 데이터 전처리 파이프라인을 정의하세요 --- #
# 튜토리얼을 참고하여 train_transform과 test_transform을 완성하세요.
# MobileNet_V2도 ResNet과 동일하게 ImageNet으로 학습되었으므로,
# 동일한 정규화 파라미터를 사용합니다.
data_transforms = {
    'train': transforms.Compose([
        # 여기에 훈련 데이터용 변환을 추가하세요 (RandomResizedCrop, RandomHorizontalFlip, ...)
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        # 여기에 검증/테스트 데이터용 변환을 추가하세요 (Resize, CenterCrop, ...)
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

image_datasets = {x: datasets.ImageFolder(f"{data_dir}/{'seg_train' if x == 'train' else 'seg_test'}", data_transforms[x]) for x in ['train', 'val']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=32, shuffle=True) for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("클래스:", class_names)
print("훈련 데이터 수:", dataset_sizes['train'])
print("검증 데이터 수:", dataset_sizes['val'])

### Part 2: 특징 추출(Feature Extraction) 모델 구축 및 학습

In [ ]:
# --- [과제] MobileNet_V2 모델을 불러오고, 특징 추출을 위해 모델을 수정하세요 --- #

# 1. `models.mobilenet_v2`를 `pretrained=True`로 불러옵니다.
# [코드를 작성하세요]

# 2. MobileNet_V2의 마지막 분류기 층은 `model.classifier`에 있습니다.
#    구조를 출력해서 확인하고, 우리 문제에 맞게 마지막 Linear 층을 교체하세요.
#    (힌트: MobileNet_V2의 마지막 층은 nn.Sequential 안에 있습니다.)
# [코드를 작성하세요]

# 3. 모델의 모든 파라미터를 동결(freeze)하세요.
# [코드를 작성하세요]

# 4. 새로 추가한 분류기 층의 파라미터만 동결을 해제(unfreeze)하세요.
# [코드를 작성하세요]

# 모델을 디바이스로 이동
# [코드를 작성하세요]

# 손실 함수 정의
# [코드를 작성하세요]

# 옵티마이저에 학습시킬 파라미터만 전달합니다.
# [코드를 작성하세요]

# --- [과제] 아래 제공된 학습 함수를 호출하여 모델을 5 에포크 동안 학습시키세요 --- #
# (학습 함수는 수정을 위해 제공됩니다. 필요시 자유롭게 수정하세요.)

def train_model(model, criterion, optimizer, num_epochs=5):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)
        for phase in ['train', 'val']:
            if phase == 'train': model.train()
            else: model.eval()

            running_loss, running_corrects = 0.0, 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
    return model, history

# --- [과제] 위 함수를 호출하여 특징 추출 모델을 학습시키세요. ---
# [코드를 작성하세요]


### Part 3: (도전 과제) 미세조정(Fine-Tuning)으로 성능 개선

In [ ]:
# --- [과제] Fine-Tuning을 위해 모델과 옵티마이저를 설정하세요 --- #

# 1. MobileNet_V2의 feature extractor 파트의 마지막 몇 개 블록을 unfreeze 하세요.
#    (예: `model_ft.features[-3:]` 의 파라미터들의 `requires_grad`를 True로 설정)
# [코드를 작성하세요]

# 2. Fine-Tuning을 위한 새로운 옵티마이저를 정의하세요.
#    (주의: 반드시 이전보다 훨씬 작은 학습률(e.g., 1e-5)을 사용하세요!)
# [코드를 작성하세요]

# 3. train_model 함수를 다시 호출하여 5 에포크 더 학습시키세요.
# [코드를 작성하세요]


### Part 4: 학습 결과 시각화 및 분석

In [ ]:
# --- [과제] Plotly를 사용하여 학습/검증 손실 및 정확도를 시각화하세요 --- #
# 1. history 딕셔너리를 사용하여 학습/검증 손실과 정확도를 시각화하세요.
# 2. plotly.express를 사용하여 두 개의 서브플롯을 만들어주세요.
# [코드를 작성하세요]

# --- [과제] 최종 모델의 예측을 사용하여 Confusion Matrix를 만드세요 --- #
# 1. final_model을 eval 모드로 설정하고 검증 데이터에 대해 예측을 수행하세요.
# 2. confusion_matrix를 계산하고 plotly.express.imshow로 시각화하세요.
# [코드를 작성하세요]

# --- [과제] Confusion Matrix 분석 --- #
# 1. Confusion Matrix를 보고 모델이 어떤 클래스들을 특히 헷갈려하는지 분석하세요.
# 2. 그 이유는 무엇일지 자신의 생각을 마크다운 셀에 작성해보세요.
# [분석을 작성하세요]